# Chapter 17 Computational Lab
## The Poisson Process

This notebook accompanies Chapter 17 of *Probability Theory with Python and AI*.

The Poisson process is the basic continuous-time counting model with a constant event rate. Its central feature is that several apparently different descriptions are equivalent:

$$
\boxed{
\text{exponential interarrival times}
}
$$

$$
\boxed{
\text{stationary independent Poisson increments}
}
$$

and

$$
\boxed{
\text{local small-time probabilities}.
}
$$

### Learning goals

By the end of the lab you should be able to:

1. define a counting process and Poisson arrival times;
2. explain why the exponential interarrival construction is non-explosive;
3. use the dual identities connecting $N(t)$ and $S_n$;
4. derive the joint density of the first $n$ arrival times;
5. recognize $S_n$ as Gamma$(n,\lambda)$ in shape--rate notation;
6. derive $N(t)\sim\operatorname{Poisson}(\lambda t)$;
7. connect Poisson counts with the classical rare-event limit;
8. distinguish stationary increments from stationarity of the process itself;
9. distinguish independent increments from independence of nested counts;
10. use the natural filtration $\mathcal F_t^N$;
11. interpret future increments as independent of the complete observed past;
12. use the continuous-time Markov transition kernel;
13. verify the semigroup property;
14. compute conditional future means;
15. understand the compensated process $N(t)-\lambda t$ as a martingale;
16. use the residual-waiting-time memorylessness property;
17. compute the covariance and correlation of nested Poisson counts;
18. use the infinitesimal generator and Kolmogorov forward equations;
19. understand the equivalence of interarrival, increment and small-time descriptions;
20. use characteristic-function uniqueness in that equivalence proof;
21. superpose independent Poisson streams;
22. analyze competing exponential clocks;
23. thin and split Poisson processes into independent subprocesses;
24. condition on $N(t)=n$ and recover uniform order statistics;
25. use Beta laws for individual conditional arrival times;
26. understand conditional gaps as a Dirichlet$(1,\ldots,1)$ simplex law;
27. prove and simulate the strong long-run rate $N(t)/t\to\lambda$;
28. construct nonhomogeneous Poisson processes by deterministic time change;
29. audit AI-generated claims about Poisson processes.

> **Important distinction.** The variables $N(s)$ and $N(t)$ are generally dependent when $s<t$. What is independent is the future increment $N(t)-N(s)$ from the past.


## 0. Setup


In [ ]:
from math import factorial
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import HTML, Math, Markdown, clear_output, display

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass


def poisson_pmf(k, mean):
    if k < 0 or int(k) != k:
        return 0.0
    k = int(k)
    if mean == 0:
        return 1.0 if k == 0 else 0.0
    return math.exp(-mean)*(mean**k)/math.factorial(k)


def simulate_poisson_arrivals(rate, horizon, rng):
    if isinstance(rate, (bool, np.bool_)) or not np.isfinite(rate) or rate <= 0:
        raise ValueError("rate must be a positive finite number")
    if isinstance(horizon, (bool, np.bool_)) or not np.isfinite(horizon) or horizon < 0:
        raise ValueError("horizon must be a non-negative finite number")

    arrivals = []
    t = 0.0

    while True:
        t += rng.exponential(1.0/rate)

        if t > horizon:
            break

        arrivals.append(t)

    return np.array(arrivals, dtype=float)


def poisson_counts_from_arrivals(arrivals, grid):
    return np.searchsorted(
        np.asarray(arrivals, dtype=float),
        np.asarray(grid, dtype=float),
        side="right",
    )


def poisson_transition(i, j, s, rate):
    if s == 0:
        return 1.0 if i == j else 0.0

    if j < i:
        return 0.0

    return poisson_pmf(j-i, rate*s)


def poisson_generator_f(f, i, rate):
    return rate*(f(i+1)-f(i))


def gamma_integer_density(s, n, rate):
    s = np.asarray(s, dtype=float)
    result = np.zeros_like(s)

    mask = s > 0
    result[mask] = (
        rate**n
        *
        s[mask]**(n-1)
        *
        np.exp(-rate*s[mask])
        /
        math.factorial(n-1)
    )

    return result


def nhpp_cumulative_intensity_linear(t, a=2.0, b=1.0):
    # intensity a + b t
    t = np.asarray(t, dtype=float)
    return a*t + 0.5*b*t*t


def simulate_nhpp_by_time_change(a, b, horizon, rng):
    Lambda_T = float(
        nhpp_cumulative_intensity_linear(
            horizon,
            a=a,
            b=b,
        )
    )

    transformed = simulate_poisson_arrivals(
        1.0,
        Lambda_T,
        rng,
    )

    if b == 0:
        return transformed/a

    # Solve a t + b t^2 / 2 = u for t >= 0.
    return (
        -a
        +
        np.sqrt(a*a + 2*b*transformed)
    )/b


def beta_integer_pdf(u, a, b):
    u = np.asarray(u, dtype=float)
    result = np.zeros_like(u)

    mask = (u > 0) & (u < 1)

    B = (
        math.factorial(a-1)
        *
        math.factorial(b-1)
        /
        math.factorial(a+b-1)
    )

    result[mask] = (
        u[mask]**(a-1)
        *
        (1-u[mask])**(b-1)
        /
        B
    )

    return result


def show_result(title, *latex_lines, note=None):
    display(HTML(
        f"<div style='border-left:5px solid;padding:8px 12px;margin:8px 0'>"
        f"<b>{title}</b></div>"
    ))

    for line in latex_lines:
        display(Math(line))

    if note:
        display(Markdown(note))


display(HTML(
    "<div style='padding:10px;border:1px solid'>"
    "<b>Setup complete.</b> Poisson-process tools are ready."
    "</div>"
))


## 1. Counting processes and arrival times

A stochastic process $(N(t))_{t\ge0}$ is a counting process when:

1. $N(0)=0$ almost surely;
2. $N(t)\in\mathbb N_0$ almost surely for every $t\ge0$;
3. sample paths are non-decreasing and right-continuous almost surely.

The value $N(t)$ counts events observed during $[0,t]$.


Let the successive waiting times be

$$
X_1,X_2,\ldots.
$$

Define

$$
S_0=0,
$$

and

$$
\boxed{
S_n=X_1+\cdots+X_n.
}
$$

Then $S_n$ is the time of the $n$th event.


### Poisson process from exponential interarrivals

For $\lambda>0$, let

$$
X_1,X_2,\ldots
\stackrel{\mathrm{i.i.d.}}{\sim}
\operatorname{Exp}(\lambda).
$$

The corresponding counting process is the rate-$\lambda$ Poisson process.

The rate-zero convention is the degenerate process

$$
\boxed{
N(t)\equiv0.
}
$$


### Non-explosion

The chapter proves that

$$
\boxed{
S_n\to\infty
\quad\text{almost surely}.
}
$$

Thus only finitely many arrivals occur in any finite time interval.

This is important: a continuous-time counting model must not accumulate infinitely many jumps before a finite time.


In [ ]:
path_rate = widgets.FloatSlider(
    value=2.0,
    min=0.2,
    max=6,
    step=0.2,
    description="lambda",
)

path_T = widgets.FloatSlider(
    value=6.0,
    min=1,
    max=20,
    step=0.5,
    description="T",
)

path_output = widgets.Output()


def update_path(*_):
    with path_output:
        clear_output(wait=True)

        lam = path_rate.value
        T = path_T.value
        rng = np.random.default_rng(2026)

        arrivals = simulate_poisson_arrivals(
            lam,
            T,
            rng,
        )

        grid = np.linspace(0,T,700)
        counts = poisson_counts_from_arrivals(
            arrivals,
            grid,
        )

        fig, ax = plt.subplots(figsize=(8,3.5))
        ax.step(
            grid,
            counts,
            where="post",
        )
        ax.set_xlabel("time")
        ax.set_ylabel("N(t)")
        ax.set_title("One simulated Poisson-process path")
        plt.show()

        display(Markdown(
            f"Number of arrivals by time $T$: **{len(arrivals)}**"
        ))


for control in (path_rate,path_T):
    control.observe(update_path, names="value")

display(widgets.VBox([
    widgets.HBox([path_rate,path_T]),
    path_output,
]))

update_path()


## 2. Duality between arrival times and counts

On the probability-one non-explosive event,

$$
\boxed{
\{N(t)\ge n\}
=
\{S_n\le t\},
}
$$

and

$$
\boxed{
\{N(t)=n\}
=
\{S_n\le t<S_{n+1}\}.
}
$$

The variables $S_n$ answer

> When does the $n$th event occur?

while $N(t)$ answers

> How many events have occurred by time $t$?


In [ ]:
rng = np.random.default_rng(2026)

arrivals = simulate_poisson_arrivals(
    2.0,
    8.0,
    rng,
)

t0 = 3.0
count = np.searchsorted(
    arrivals,
    t0,
    side="right",
)

display(Markdown(f"Arrival times: **{np.round(arrivals,3)}**"))
display(Math(r"N(3)=" + f"{count}"))

if count >= 1:
    display(Math(
        r"S_{N(3)}\le3<S_{N(3)+1}"
    ))


## 3. Joint density of the first arrival times

For independent Exp$(\lambda)$ interarrivals,

$$
\boxed{
f_{S_1,\ldots,S_n}
(s_1,\ldots,s_n)
=
\lambda^n
e^{-\lambda s_n}
\mathbf1_{\{0<s_1<\cdots<s_n\}}.
}
$$

The change of variables from interarrival times to arrival times is triangular and has Jacobian determinant one.


### Distribution of the $n$th arrival time

Integrating over the ordered simplex gives

$$
\boxed{
S_n
\sim
\operatorname{Gamma}(n,\lambda)
}
$$

in shape--rate notation, with density

$$
\boxed{
f_{S_n}(s)
=
\frac{\lambda^n}{(n-1)!}
s^{n-1}
e^{-\lambda s},
\qquad
s>0.
}
$$


### Fourth arrival in a rate-$2$ process

$$
S_4
\sim
\operatorname{Gamma}(4,2),
$$

so

$$
\boxed{
\mathbb E[S_4]=2,
\qquad
\operatorname{Var}(S_4)=1.
}
$$


In [ ]:
gamma_n = widgets.IntSlider(
    value=4,
    min=1,
    max=10,
    description="n",
)

gamma_rate = widgets.FloatSlider(
    value=2,
    min=0.5,
    max=6,
    step=0.25,
    description="lambda",
)

gamma_output = widgets.Output()


def update_gamma_arrival(*_):
    with gamma_output:
        clear_output(wait=True)

        n = gamma_n.value
        lam = gamma_rate.value

        mean = n/lam
        var = n/(lam*lam)

        s = np.linspace(
            0,
            max(1, mean+4*math.sqrt(var)),
            800,
        )

        density = gamma_integer_density(
            s,
            n,
            lam,
        )

        fig, ax = plt.subplots(figsize=(8,3.3))
        ax.plot(s,density)
        ax.set_xlabel("s")
        ax.set_ylabel("density")
        ax.set_title("Density of the nth Poisson arrival time")
        plt.show()

        display(Math(r"\mathbb E[S_n]=" + f"{mean:.6f}"))
        display(Math(r"\operatorname{Var}(S_n)=" + f"{var:.6f}"))


for control in (gamma_n,gamma_rate):
    control.observe(update_gamma_arrival, names="value")

display(widgets.VBox([
    widgets.HBox([gamma_n,gamma_rate]),
    gamma_output,
]))

update_gamma_arrival()


## 4. Why the Poisson distribution appears

For every $t\ge0$,

$$
\boxed{
N(t)
\sim
\operatorname{Poisson}(\lambda t).
}
$$

For $t>0$,

$$
\boxed{
P(N(t)=n)
=
e^{-\lambda t}
\frac{(\lambda t)^n}{n!}.
}
$$

At $t=0$, the process is zero almost surely.


### Mean and variance

Since

$$
N(t)
\sim
\operatorname{Poisson}(\lambda t),
$$

$$
\boxed{
\mathbb E[N(t)]
=
\lambda t,
}
$$

and

$$
\boxed{
\operatorname{Var}(N(t))
=
\lambda t.
}
$$


In [ ]:
count_lam = widgets.FloatSlider(
    value=2,
    min=0.5,
    max=6,
    step=0.25,
    description="lambda",
)

count_t = widgets.FloatSlider(
    value=2,
    min=0.2,
    max=6,
    step=0.2,
    description="t",
)

count_output = widgets.Output()


def update_count_law(*_):
    with count_output:
        clear_output(wait=True)

        lam = count_lam.value
        t = count_t.value
        mean = lam*t

        max_k = max(
            8,
            int(mean + 5*math.sqrt(mean) + 2),
        )

        ks = np.arange(max_k+1)
        pmf = np.array([
            poisson_pmf(int(k),mean)
            for k in ks
        ])

        fig, ax = plt.subplots(figsize=(8,3.4))
        ax.bar(ks,pmf)
        ax.set_xlabel("k")
        ax.set_ylabel("P(N(t)=k)")
        ax.set_title("Poisson count distribution")
        plt.show()

        display(Math(r"\mathbb E[N(t)]=" + f"{mean:.6f}"))
        display(Math(r"\operatorname{Var}(N(t))=" + f"{mean:.6f}"))


for control in (count_lam,count_t):
    control.observe(update_count_law, names="value")

display(widgets.VBox([
    widgets.HBox([count_lam,count_t]),
    count_output,
]))

update_count_law()


### Ten arrivals in half an hour

If

$$
\lambda=12
$$

per hour, then

$$
N(1/2)
\sim
\operatorname{Poisson}(6).
$$

Therefore

$$
\boxed{
P(N(1/2)=10)
=
e^{-6}
\frac{6^{10}}{10!}.
}
$$


In [ ]:
p10 = poisson_pmf(10,6)
display(Math(r"P(N(1/2)=10)=" + f"{p10:.8f}"))


## 5. Historical problem: rare events become Poisson

Divide a fixed interval of length $t$ into $m$ very short subintervals.

Suppose each subinterval independently contains an event with probability

$$
p_m
=
\frac{\lambda t}{m}.
$$

Then

$$
Y_m
\sim
\operatorname{Bin}
\left(
m,
\frac{\lambda t}{m}
\right).
$$

For each fixed $k$,

$$
\boxed{
P(Y_m=k)
\longrightarrow
e^{-\lambda t}
\frac{(\lambda t)^k}{k!}.
}
$$


In [ ]:
rare_m = widgets.IntSlider(
    value=50,
    min=5,
    max=1000,
    step=5,
    description="m",
)

rare_mean = widgets.FloatSlider(
    value=3.0,
    min=0.5,
    max=8,
    step=0.5,
    description="lambda*t",
)

rare_output = widgets.Output()


def update_rare_limit(*_):
    with rare_output:
        clear_output(wait=True)

        m = rare_m.value
        theta = rare_mean.value

        if theta > m:
            display(Markdown("Increase $m$ so that $\\lambda t/m\\le1$."))
            return

        p = theta/m
        max_k = min(m, int(theta+5*math.sqrt(theta)+5))
        ks = np.arange(max_k+1)

        binom = np.array([
            math.comb(m,int(k))
            *
            p**int(k)
            *
            (1-p)**(m-int(k))
            for k in ks
        ])

        pois = np.array([
            poisson_pmf(int(k),theta)
            for k in ks
        ])

        fig, ax = plt.subplots(figsize=(8,3.5))
        ax.plot(ks,binom,"o-",label="Binomial rare-event model")
        ax.plot(ks,pois,"s--",label="Poisson limit")
        ax.set_xlabel("k")
        ax.set_ylabel("probability")
        ax.legend()
        ax.set_title("Rare-event binomial to Poisson limit")
        plt.show()

        display(Markdown(
            f"Maximum displayed mass difference: **{np.max(np.abs(binom-pois)):.6g}**"
        ))


for control in (rare_m,rare_mean):
    control.observe(update_rare_limit, names="value")

display(widgets.VBox([
    widgets.HBox([rare_m,rare_mean]),
    rare_output,
]))

update_rare_limit()


### Historical application: Erlang and telephone traffic

Suppose calls follow a homogeneous Poisson process at rate

$$
30
$$

per hour.

In two minutes, the expected count is one, so

$$
\boxed{
P(N(2\text{ min})=0)
=
e^{-1}.
}
$$

The fifth arrival time satisfies

$$
S_5
\sim
\operatorname{Gamma}(5,30)
$$

when time is measured in hours, and therefore

$$
\boxed{
\mathbb E[S_5]
=
\frac5{30}\text{ hour}
=
10\text{ minutes}.
}
$$

A constant-rate model may be reasonable over a short period but unrealistic across an entire day.


## 6. Stationary and independent increments

A process has **stationary increments** if

$$
X(t+s)-X(t)
$$

depends in distribution on $s$ but not on $t$.

A process has **independent increments** when increments over disjoint intervals are mutually independent.


For the Poisson process,

$$
\boxed{
N(t+s)-N(t)
\sim
\operatorname{Poisson}(\lambda s),
}
$$

and increments over disjoint intervals are independent.


### Stationary increments do not mean a stationary process

Since

$$
N(t)
\sim
\operatorname{Poisson}(\lambda t),
$$

the law of $N(t)$ changes with time.

Thus $N$ is **not** a stationary process.

Only its increments are stationary.


In [ ]:
inc_N = widgets.IntSlider(
    value=30000,
    min=2000,
    max=100000,
    step=2000,
    description="reps",
)

inc_output = widgets.Output()


def update_increment_independence(*_):
    with inc_output:
        clear_output(wait=True)

        reps = inc_N.value
        rng = np.random.default_rng(2026)

        inc1 = rng.poisson(2,size=reps)
        inc2 = rng.poisson(3,size=reps)

        corr = np.corrcoef(
            inc1,
            inc2,
        )[0,1]

        display(Math(
            r"\widehat{\operatorname{Corr}}(\Delta_1N,\Delta_2N)="
            + f"{corr:.6f}"
        ))


inc_N.observe(update_increment_independence, names="value")
display(widgets.VBox([inc_N,inc_output]))
update_increment_independence()


## 7. Natural filtration and independence from the past

The natural filtration is

$$
\boxed{
\mathcal F_t^N
=
\sigma(
N(s):0\le s\le t
).
}
$$

It contains all information revealed by the observed counting path up to time $t$.


For every $t,h\ge0$,

$$
\boxed{
N(t+h)-N(t)
\perp\!\!\!\perp
\mathcal F_t,
}
$$

and

$$
N(t+h)-N(t)
\sim
\operatorname{Poisson}(\lambda h).
$$

This statement is stronger than saying that the increment is independent of $N(t)$ alone.


The chapter extends independence from finite collections of past counts to the full past filtration by a $\pi$--$\lambda$ argument.


## 8. The Poisson process as a continuous-time Markov process

The transition kernel is

$$
\boxed{
p_{ij}(s)
=
P(
N(t+s)=j
\mid
N(t)=i
)
=
e^{-\lambda s}
\frac{(\lambda s)^{j-i}}{(j-i)!}
}
$$

for $s>0$ and $j\ge i$, and zero when $j<i$.


For bounded $g$,

$$
\boxed{
\mathbb E[
g(N(t+s))
\mid
\mathcal F_t
]
=
\sum_{r=0}^{\infty}
g(N(t)+r)
e^{-\lambda s}
\frac{(\lambda s)^r}{r!}.
}
$$

The complete past enters only through the current count $N(t)$.


### Example

For rate $2$, if

$$
N(3)=5,
$$

then

$$
P(N(4)=7\mid N(3)=5)
=
P(
\operatorname{Poisson}(2)=2
)
=
\boxed{
2e^{-2}.
}
$$


In [ ]:
display(Math(
    r"P(N(4)=7\mid N(3)=5)="
    + f"{poisson_transition(5,7,1,2):.8f}"
))


## 9. Semigroup property

Let

$$
P(s)
=
(p_{ij}(s))_{i,j\in\mathbb N_0}.
$$

Then

$$
\boxed{
P(s+t)
=
P(s)P(t).
}
$$

Entrywise,

$$
\boxed{
p_{ij}(s+t)
=
\sum_k
p_{ik}(s)
p_{kj}(t).
}
$$

For fixed $i,j$, the sum is finite because the process can only move upward.


In [ ]:
i,j = 2,7
s,t = 0.7,1.3
lam = 2.5

lhs = poisson_transition(
    i,j,s+t,lam
)

rhs = sum(
    poisson_transition(i,k,s,lam)
    *
    poisson_transition(k,j,t,lam)
    for k in range(i,j+1)
)

display(Math(r"p_{2,7}(2.0)=" + f"{lhs:.10f}"))
display(Math(r"\sum_kp_{2k}(0.7)p_{k7}(1.3)=" + f"{rhs:.10f}"))


## 10. Conditional mean and compensated martingale

For

$$
0\le s\le t,
$$

independent increments give

$$
\boxed{
\mathbb E[
N(t)
\mid
\mathcal F_s
]
=
N(s)
+
\lambda(t-s).
}
$$

The best conditional prediction is the observed count plus the expected number of future arrivals.


Define

$$
\boxed{
M(t)
=
N(t)-\lambda t.
}
$$

Then

$$
\boxed{
\mathbb E[
M(t)
\mid
\mathcal F_s
]
=
M(s).
}
$$

Thus the compensated Poisson process is a martingale.


### Predicting future counts

If

$$
\lambda=5
$$

per hour and the observed path satisfies

$$
N(2)=7,
$$

then

$$
\boxed{
\mathbb E[
N(3.5)
\mid
\mathcal F_2
]
=
7+5(1.5)
=
14.5.
}
$$


In [ ]:
future_mean = 7 + 5*(3.5-2)
display(Math(
    r"\mathbb E[N(3.5)\mid\mathcal F_2]="
    + f"{future_mean:.1f}"
))


## 11. Residual waiting time

Let $R_t$ be the waiting time from a fixed observation time $t$ until the next arrival.

Then

$$
\boxed{
R_t
\sim
\operatorname{Exp}(\lambda),
}
$$

and

$$
\boxed{
R_t
\perp\!\!\!\perp
\mathcal F_t.
}
$$

At any fixed observation time, the residual wait is a fresh exponential random variable independent of the complete observed past.


For a rate-$12$ process per hour,

$$
\boxed{
\mathbb E[R_t]
=
\frac1{12}\text{ hour}
=
5\text{ minutes}.
}
$$


In [ ]:
residual_lam = widgets.FloatSlider(
    value=12,
    min=1,
    max=30,
    step=1,
    description="lambda/hour",
)
residual_output = widgets.Output()


def update_residual(*_):
    with residual_output:
        clear_output(wait=True)

        lam = residual_lam.value

        display(Math(
            r"\mathbb E[R_t]\text{ in minutes}="
            + f"{60/lam:.4f}"
        ))


residual_lam.observe(update_residual, names="value")
display(widgets.VBox([residual_lam,residual_output]))
update_residual()


## 12. Covariance structure

For $s,t\ge0$,

$$
\boxed{
\operatorname{Cov}
(
N(s),N(t)
)
=
\lambda\min(s,t).
}
$$

For $s,t>0$,

$$
\boxed{
\operatorname{Corr}
(
N(s),N(t)
)
=
\sqrt{
\frac{
\min(s,t)
}{
\max(s,t)
}
}.
}
$$


If $0<s\le t$,

$$
N(t)
=
N(s)
+
[N(t)-N(s)],
$$

and the bracketed increment is independent of $N(s)$.

Hence

$$
\operatorname{Cov}(N(s),N(t))
=
\operatorname{Var}(N(s))
=
\lambda s.
$$


In [ ]:
cov_s = widgets.FloatSlider(
    value=1,
    min=0.1,
    max=5,
    step=0.1,
    description="s",
)
cov_t = widgets.FloatSlider(
    value=4,
    min=0.1,
    max=8,
    step=0.1,
    description="t",
)
cov_lam = widgets.FloatSlider(
    value=3,
    min=0.5,
    max=8,
    step=0.5,
    description="lambda",
)
cov_output = widgets.Output()


def update_covariance(*_):
    with cov_output:
        clear_output(wait=True)

        s = cov_s.value
        t = cov_t.value
        lam = cov_lam.value

        covariance = lam*min(s,t)
        corr = math.sqrt(
            min(s,t)/max(s,t)
        )

        display(Math(r"\operatorname{Cov}=" + f"{covariance:.6f}"))
        display(Math(r"\operatorname{Corr}=" + f"{corr:.6f}"))


for control in (cov_s,cov_t,cov_lam):
    control.observe(update_covariance, names="value")

display(widgets.VBox([
    widgets.HBox([cov_s,cov_t,cov_lam]),
    cov_output,
]))

update_covariance()


## 13. Infinitesimal generator

For suitable $f:\mathbb N_0\to\mathbb R$,

$$
(Af)(i)
=
\lim_{h\downarrow0}
\frac{
\sum_jp_{ij}(h)f(j)-f(i)
}{
h
}.
$$

For every bounded $f$,

$$
\boxed{
(Af)(i)
=
\lambda[
f(i+1)-f(i)
].
}
$$


The generator matrix has

$$
\boxed{
q_{ii}
=
-\lambda,
}
$$

$$
\boxed{
q_{i,i+1}
=
\lambda,
}
$$

and all other entries zero.

To first order, the only possible move is one upward jump.


### Generator applied to the count

For

$$
f(i)=i,
$$

$$
\boxed{
(Af)(i)
=
\lambda.
}
$$

For

$$
f(i)=i^2,
$$

$$
\boxed{
(Af)(i)
=
\lambda(2i+1).
}
$$


In [ ]:
gen_i = widgets.IntSlider(
    value=4,
    min=0,
    max=20,
    description="i",
)
gen_lam = widgets.FloatSlider(
    value=2,
    min=0.2,
    max=6,
    step=0.2,
    description="lambda",
)
gen_output = widgets.Output()


def update_generator(*_):
    with gen_output:
        clear_output(wait=True)

        i = gen_i.value
        lam = gen_lam.value

        linear = poisson_generator_f(
            lambda k:k,
            i,
            lam,
        )

        square = poisson_generator_f(
            lambda k:k*k,
            i,
            lam,
        )

        display(Math(r"A(i)=" + f"{linear:.6f}"))
        display(Math(r"A(i^2)=" + f"{square:.6f}"))


for control in (gen_i,gen_lam):
    control.observe(update_generator, names="value")

display(widgets.VBox([
    widgets.HBox([gen_i,gen_lam]),
    gen_output,
]))

update_generator()


## 14. Kolmogorov forward equations

For $j\ge i$,

$$
\boxed{
\frac{d}{dt}
p_{ij}(t)
=
\lambda
p_{i,j-1}(t)
-
\lambda
p_{ij}(t),
}
$$

with

$$
p_{i,i-1}(t)=0.
$$

For

$$
p_n(t)
=
P(N(t)=n),
$$

$$
\boxed{
p_0'(t)
=
-\lambda p_0(t),
}
$$

and

$$
\boxed{
p_n'(t)
=
\lambda p_{n-1}(t)
-
\lambda p_n(t),
\qquad
n\ge1.
}
$$


In [ ]:
fd_n = 3
fd_t = 1.2
fd_lam = 2.0
h = 1e-5

p_plus = poisson_pmf(fd_n,fd_lam*(fd_t+h))
p_minus = poisson_pmf(fd_n,fd_lam*(fd_t-h))
num_derivative = (p_plus-p_minus)/(2*h)

rhs = (
    fd_lam*poisson_pmf(fd_n-1,fd_lam*fd_t)
    -
    fd_lam*poisson_pmf(fd_n,fd_lam*fd_t)
)

display(Math(r"\text{numerical derivative}=" + f"{num_derivative:.10f}"))
display(Math(r"\lambda p_{n-1}-\lambda p_n=" + f"{rhs:.10f}"))


## 15. Small-time behavior

As $h\downarrow0$,

$$
\boxed{
P(N(h)=0)
=
1-\lambda h+o(h),
}
$$

$$
\boxed{
P(N(h)=1)
=
\lambda h+o(h),
}
$$

and consequently

$$
\boxed{
P(N(h)\ge2)
=
o(h).
}
$$

The chance of two or more events is second order in the interval length.


In [ ]:
small_lam = 3.0
hs = np.logspace(-4,-1,100)

p0_error = np.array([
    abs(
        poisson_pmf(0,small_lam*h)
        -
        (1-small_lam*h)
    )
    for h in hs
])

pge2 = np.array([
    1
    -
    poisson_pmf(0,small_lam*h)
    -
    poisson_pmf(1,small_lam*h)
    for h in hs
])

fig, ax = plt.subplots(figsize=(8,3.5))
ax.loglog(hs,p0_error,label="|P(N(h)=0)-(1-lambda h)|")
ax.loglog(hs,pge2,label="P(N(h)>=2)")
ax.set_xlabel("h")
ax.set_ylabel("size")
ax.set_title("Small-time Poisson terms")
ax.legend()
plt.show()


## 16. Three equivalent descriptions

For $\lambda>0$, the chapter proves equivalence in finite-dimensional law among:

1. independent Exp$(\lambda)$ interarrival times;
2. independent Poisson increments with parameter $\lambda$ times interval length;
3. stationary independent increments plus the small-time rules.

The difficult direction is the small-time rule $\Rightarrow$ full Poisson increment law.


### Why characteristic-function uniqueness matters

Let

$$
Y_s
=
N(t+s)-N(t).
$$

The small-time conditions imply

$$
\varphi_h(u)
=
1
+
\lambda h(e^{iu}-1)
+
o(h).
$$

Independent increments give the semigroup relation

$$
\varphi_{s+h}(u)
=
\varphi_s(u)\varphi_h(u).
$$

Hence

$$
\frac{d}{ds}
\varphi_s(u)
=
\lambda(e^{iu}-1)
\varphi_s(u),
$$

with $\varphi_0(u)=1$.


The ODE solution is

$$
\boxed{
\varphi_s(u)
=
\exp\{
\lambda s(e^{iu}-1)
\}.
}
$$

This is the characteristic function of Poisson$(\lambda s)$.

Characteristic-function uniqueness therefore identifies the full increment distribution.


## 17. Superposition

If

$$
N_1
$$

and

$$
N_2
$$

are independent Poisson processes with rates $\lambda_1$ and $\lambda_2$, then

$$
\boxed{
N(t)
=
N_1(t)+N_2(t)
}
$$

is a Poisson process with rate

$$
\boxed{
\lambda_1+\lambda_2.
}
$$


### Two streams

Rates $2$ and $5$ per hour combine to rate $7$ per hour.

Over half an hour,

$$
N(1/2)
\sim
\operatorname{Poisson}(7/2).
$$


In [ ]:
super_N = widgets.IntSlider(
    value=30000,
    min=2000,
    max=100000,
    step=2000,
    description="reps",
)
super_output = widgets.Output()


def update_superposition(*_):
    with super_output:
        clear_output(wait=True)

        reps = super_N.value
        rng = np.random.default_rng(2026)

        a = rng.poisson(2*0.5,size=reps)
        b = rng.poisson(5*0.5,size=reps)
        total = a+b

        display(Math(r"\widehat{\mathbb E}[N(1/2)]=" + f"{total.mean():.6f}"))
        display(Math(r"\text{theory}=3.5"))
        display(Math(r"\widehat{\operatorname{Var}}=" + f"{total.var():.6f}"))


super_N.observe(update_superposition, names="value")
display(widgets.VBox([super_N,super_output]))
update_superposition()


## 18. Competing exponential clocks

Let

$$
X_j
\sim
\operatorname{Exp}(\lambda_j)
$$

independently.

For

$$
M
=
\min_jX_j,
$$

$$
\boxed{
M
\sim
\operatorname{Exp}
\left(
\sum_j\lambda_j
\right).
}
$$

Also,

$$
\boxed{
P(M=X_k)
=
\frac{
\lambda_k
}{
\sum_j\lambda_j
}.
}
$$


For rates $3$ and $7$,

$$
\boxed{
M\sim\operatorname{Exp}(10),
}
$$

and the next event is from the first stream with probability

$$
\boxed{
0.3.
}
$$


In [ ]:
clock_N = widgets.IntSlider(
    value=50000,
    min=2000,
    max=200000,
    step=2000,
    description="reps",
)
clock_output = widgets.Output()


def update_competing(*_):
    with clock_output:
        clear_output(wait=True)

        N = clock_N.value
        rng = np.random.default_rng(2026)

        x1 = rng.exponential(1/3,size=N)
        x2 = rng.exponential(1/7,size=N)

        minimum = np.minimum(x1,x2)
        first_wins = np.mean(x1<x2)

        display(Math(r"\widehat P(X_1<X_2)=" + f"{first_wins:.6f}"))
        display(Math(r"\text{theory}=0.3"))
        display(Math(r"\widehat{\mathbb E}[M]=" + f"{minimum.mean():.6f}"))
        display(Math(r"\text{theory}=0.1"))


clock_N.observe(update_competing, names="value")
display(widgets.VBox([clock_N,clock_output]))
update_competing()


## 19. Poisson thinning and splitting

Start with a rate-$\lambda$ Poisson process.

Independently mark each event as retained with probability $p$.

Then the retained process is Poisson with rate

$$
\boxed{
p\lambda,
}
$$

the discarded process is Poisson with rate

$$
\boxed{
(1-p)\lambda,
}
$$

and the two subprocesses are independent.


More generally, if events receive independent labels with probabilities

$$
p_1,\ldots,p_m,
$$

then the type-specific counting processes are mutually independent Poisson processes with rates

$$
\boxed{
p_1\lambda,\ldots,p_m\lambda.
}
$$


In [ ]:
thin_N = widgets.IntSlider(
    value=30000,
    min=2000,
    max=100000,
    step=2000,
    description="reps",
)
thin_p = widgets.FloatSlider(
    value=0.3,
    min=0,
    max=1,
    step=0.05,
    description="p",
)
thin_output = widgets.Output()


def update_thinning(*_):
    with thin_output:
        clear_output(wait=True)

        reps = thin_N.value
        p = thin_p.value

        rng = np.random.default_rng(2026)

        lam = 2.0
        t = 2.0

        base = rng.poisson(
            lam*t,
            size=reps,
        )

        kept = rng.binomial(
            base,
            p,
        )

        removed = base-kept

        corr = np.corrcoef(
            kept,
            removed,
        )[0,1]

        display(Math(
            r"\widehat{\mathbb E}[N_{\mathrm{keep}}]="
            + f"{kept.mean():.6f}"
        ))
        display(Math(
            r"p\lambda t="
            + f"{p*lam*t:.6f}"
        ))
        display(Math(
            r"\widehat{\operatorname{Corr}}(N_{\mathrm{keep}},N_{\mathrm{remove}})="
            + f"{corr:.6f}"
        ))


for control in (thin_N,thin_p):
    control.observe(update_thinning, names="value")

display(widgets.VBox([
    widgets.HBox([thin_N,thin_p]),
    thin_output,
]))

update_thinning()


## 20. Arrival times conditional on the total count

Condition on

$$
N(t)=n,
\qquad
n\ge1.
$$

Then

$$
\boxed{
f_{S_1,\ldots,S_n\mid N(t)=n}
(s_1,\ldots,s_n)
=
\frac{n!}{t^n}
\mathbf1_{\{0<s_1<\cdots<s_n<t\}}.
}
$$

This is exactly the joint density of the order statistics of $n$ independent $U(0,t)$ variables.


A crucial feature is that $\lambda$ disappears after conditioning on the total count.

Once we know that exactly $n$ events occurred in $[0,t]$, their relative locations are uniformly distributed over the ordered simplex.


### Three arrivals in ten time units

Given

$$
N(10)=3,
$$

$$
(S_1,S_2,S_3)
\stackrel d=
(U_{(1)},U_{(2)},U_{(3)}),
$$

where

$$
U_i
\stackrel{\mathrm{i.i.d.}}{\sim}
U(0,10).
$$

The conditional joint density equals

$$
\boxed{
\frac{3!}{10^3}
=
0.006
}
$$

on

$$
0<s_1<s_2<s_3<10.
$$


In [ ]:
order_n = widgets.IntSlider(
    value=3,
    min=1,
    max=10,
    description="n",
)
order_t = widgets.FloatSlider(
    value=10,
    min=1,
    max=20,
    step=1,
    description="t",
)
order_output = widgets.Output()


def update_order_stats(*_):
    with order_output:
        clear_output(wait=True)

        n = order_n.value
        t = order_t.value

        rng = np.random.default_rng(2026)
        U = np.sort(
            rng.uniform(
                0,
                t,
                size=n,
            )
        )

        display(Markdown(
            f"One exact conditional simulation by sorted uniforms: **{np.round(U,4)}**"
        ))


for control in (order_n,order_t):
    control.observe(update_order_stats, names="value")

display(widgets.VBox([
    widgets.HBox([order_n,order_t]),
    order_output,
]))

update_order_stats()


## 21. Beta law of a conditional arrival time

For

$$
k=1,\ldots,n,
$$

$$
\boxed{
\frac{S_k}{t}
\mid
\{N(t)=n\}
\sim
\operatorname{Beta}
(k,n+1-k).
}
$$

Therefore

$$
\boxed{
\mathbb E[
S_k
\mid
N(t)=n
]
=
\frac{k}{n+1}t.
}
$$


The conditional density is

$$
\boxed{
f_{S_k\mid N(t)=n}(s)
=
\frac{n!}{(k-1)!(n-k)!}
\frac{
s^{k-1}(t-s)^{n-k}
}{
t^n
},
\qquad
0<s<t.
}
$$


In [ ]:
beta_n = widgets.IntSlider(
    value=5,
    min=1,
    max=12,
    description="n",
)
beta_k = widgets.IntSlider(
    value=2,
    min=1,
    max=5,
    description="k",
)
beta_output = widgets.Output()


def update_beta_arrival(*_):
    with beta_output:
        clear_output(wait=True)

        n = beta_n.value

        if beta_k.max != n:
            beta_k.max = n

        k = min(beta_k.value,n)
        a = k
        b = n+1-k

        u = np.linspace(0,1,700)
        f = beta_integer_pdf(u,a,b)

        fig, ax = plt.subplots(figsize=(8,3.3))
        ax.plot(u,f)
        ax.set_xlabel("s/t")
        ax.set_ylabel("density")
        ax.set_title("Conditional normalized kth arrival time")
        plt.show()

        display(Math(
            r"\mathbb E[S_k/t\mid N(t)=n]="
            + f"{k/(n+1):.6f}"
        ))


for control in (beta_n,beta_k):
    control.observe(update_beta_arrival, names="value")

display(widgets.VBox([
    widgets.HBox([beta_n,beta_k]),
    beta_output,
]))

update_beta_arrival()


### One arrival given exactly one event

If

$$
N(t)=1,
$$

then

$$
\boxed{
S_1
\mid
\{N(t)=1\}
\sim
U(0,t).
}
$$


## 22. Conditional gaps are uniform on a simplex

Condition on

$$
N(t)=n.
$$

Define

$$
G_0=S_1,
$$

$$
G_k=S_{k+1}-S_k,
\qquad
1\le k\le n-1,
$$

and

$$
G_n=t-S_n.
$$

Then

$$
G_0+\cdots+G_n=t.
$$


The normalized gap vector satisfies

$$
\boxed{
\left(
\frac{G_0}{t},
\ldots,
\frac{G_n}{t}
\right)
\mid
\{N(t)=n\}
\sim
\operatorname{Dirichlet}
(1,\ldots,1).
}
$$

Equivalently, it is uniform on the simplex.

By symmetry,

$$
\boxed{
\mathbb E[
G_k
\mid
N(t)=n
]
=
\frac{t}{n+1}.
}
$$


The gaps are **not independent** because their sum is fixed at $t$.


In [ ]:
rng = np.random.default_rng(2026)

n = 3
t = 8

u = np.sort(
    rng.uniform(0,t,size=n)
)

gaps = np.diff(
    np.concatenate(
        ([0.0],u,[t])
    )
)

display(Markdown(f"Ordered arrivals: **{np.round(u,4)}**"))
display(Markdown(f"Gaps: **{np.round(gaps,4)}**"))
display(Math(r"\sum_kG_k=" + f"{gaps.sum():.6f}"))
display(Math(r"\mathbb E[G_k\mid N(8)=3]=2"))


## 23. Strong long-run event rate

For a rate-$\lambda$ Poisson process,

$$
\boxed{
\frac{N(t)}{t}
\xrightarrow[t\to\infty]{\mathrm{a.s.}}
\lambda.
}
$$

The rate is therefore both:

- the expected number of arrivals per unit time;
- the almost-sure long-run empirical event frequency.


The proof uses the strong law for interarrival times:

$$
\frac{S_n}{n}
\to
\frac1\lambda
\quad
\text{a.s.}
$$

If

$$
n=N(t),
$$

then

$$
S_n
\le
t
<
S_{n+1}.
$$

A squeeze argument gives

$$
\frac{t}{N(t)}
\to
\frac1\lambda,
$$

hence

$$
N(t)/t\to\lambda.
$$


In [ ]:
long_T = widgets.FloatSlider(
    value=250,
    min=20,
    max=500,
    step=10,
    description="T",
)
long_output = widgets.Output()


def update_long_run(*_):
    with long_output:
        clear_output(wait=True)

        T = long_T.value
        lam = 2.0

        rng = np.random.default_rng(2026)

        arrivals = simulate_poisson_arrivals(
            lam,
            T,
            rng,
        )

        grid = np.linspace(1,T,800)

        counts = poisson_counts_from_arrivals(
            arrivals,
            grid,
        )

        rate_path = counts/grid

        fig, ax = plt.subplots(figsize=(8,3.5))
        ax.plot(grid,rate_path,label="N(t)/t")
        ax.axhline(lam,linestyle="--",label="lambda")
        ax.set_xlabel("t")
        ax.set_ylabel("empirical rate")
        ax.set_title("Strong-law stabilization of the Poisson rate")
        ax.legend()
        plt.show()

        display(Math(
            r"\frac{N(T)}T="
            + f"{rate_path[-1]:.6f}"
        ))


long_T.observe(update_long_run, names="value")
display(widgets.VBox([long_T,long_output]))
update_long_run()


Almost-sure convergence immediately implies the weak law

$$
\boxed{
P\left(
\left|
\frac{N(t)}t-\lambda
\right|
>
\varepsilon
\right)
\to0.
}
$$


## 24. Nonhomogeneous Poisson process

Let

$$
\lambda(t)
\ge0
$$

be locally integrable and define

$$
\boxed{
\Lambda(t)
=
\int_0^t
\lambda(u)\,du.
}
$$

A nonhomogeneous Poisson process has independent increments with

$$
\boxed{
N(t)-N(s)
\sim
\operatorname{Poisson}
(
\Lambda(t)-\Lambda(s)
).
}
$$

Unless $\lambda(t)$ is constant, increments are generally **not** stationary.


### Time-change construction

Let

$$
\widetilde N
$$

be a unit-rate homogeneous Poisson process.

Then

$$
\boxed{
N(t)
=
\widetilde N(
\Lambda(t)
)
}
$$

is a nonhomogeneous Poisson process with intensity $\lambda(t)$.

In particular,

$$
\boxed{
N(t)
\sim
\operatorname{Poisson}(
\Lambda(t)
).
}
$$


### Example: $\lambda(t)=2+t$

Then

$$
\Lambda(t)
=
2t+\frac{t^2}{2}.
$$

Therefore

$$
N(3)
\sim
\operatorname{Poisson}(10.5),
$$

and

$$
N(3)-N(1)
\sim
\operatorname{Poisson}(8).
$$

These two variables are not independent because the first contains the earlier count $N(1)$ plus the increment from $1$ to $3$.

However, increments over **disjoint** intervals remain independent.


In [ ]:
Lambda3 = 2*3+3**2/2
Lambda1 = 2*1+1**2/2

display(Math(r"\Lambda(3)=" + f"{Lambda3:.1f}"))
display(Math(r"\Lambda(3)-\Lambda(1)=" + f"{Lambda3-Lambda1:.1f}"))


In [ ]:
nhpp_T = widgets.FloatSlider(
    value=6,
    min=1,
    max=10,
    step=0.5,
    description="T",
)
nhpp_output = widgets.Output()


def update_nhpp(*_):
    with nhpp_output:
        clear_output(wait=True)

        T = nhpp_T.value
        rng = np.random.default_rng(2026)

        arrivals = simulate_nhpp_by_time_change(
            2.0,
            1.0,
            T,
            rng,
        )

        grid = np.linspace(0,T,700)

        counts = poisson_counts_from_arrivals(
            arrivals,
            grid,
        )

        fig, ax = plt.subplots(figsize=(8,3.5))
        ax.step(
            grid,
            counts,
            where="post",
        )
        ax.set_xlabel("time")
        ax.set_ylabel("N(t)")
        ax.set_title("Nonhomogeneous Poisson process with intensity 2+t")
        plt.show()

        display(Math(
            r"\Lambda(T)="
            + f"{nhpp_cumulative_intensity_linear(T):.6f}"
        ))


nhpp_T.observe(update_nhpp, names="value")
display(widgets.VBox([nhpp_T,nhpp_output]))
update_nhpp()


### Covariance for a nonhomogeneous Poisson process

Exactly the same nested-count argument gives

$$
\boxed{
\operatorname{Cov}
(
N(s),N(t)
)
=
\Lambda(
\min(s,t)
).
}
$$

For the homogeneous process, $\Lambda(u)=\lambda u$, recovering the earlier formula.


## 25. Python laboratory: one coherent Poisson-process experiment

The next experiment reproduces the chapter's main numerical checks:

1. one sample path;
2. Poisson count law;
3. independent increments;
4. thinning;
5. long-run rate;
6. covariance;
7. superposition;
8. conditional uniform arrival times;
9. deterministic time change.


In [ ]:
lab_N = widgets.IntSlider(
    value=30000,
    min=5000,
    max=100000,
    step=5000,
    description="reps",
)
lab_output = widgets.Output()


def update_full_poisson_lab(*_):
    with lab_output:
        clear_output(wait=True)

        reps = lab_N.value
        rng = np.random.default_rng(12345)

        lam = 2.0
        t0 = 2.0

        # Count law.
        samples = rng.poisson(
            lam*t0,
            size=reps,
        )

        empirical_mean = samples.mean()
        empirical_var = samples.var()

        # Independent increments.
        inc1 = rng.poisson(
            lam,
            size=reps,
        )

        inc2 = rng.poisson(
            lam,
            size=reps,
        )

        inc_corr = np.corrcoef(
            inc1,
            inc2,
        )[0,1]

        # Thinning.
        p = 0.3

        base = rng.poisson(
            lam*t0,
            size=reps,
        )

        kept = rng.binomial(
            base,
            p,
        )

        removed = base-kept

        thin_corr = np.corrcoef(
            kept,
            removed,
        )[0,1]

        # Covariance of nested counts.
        A = rng.poisson(
            lam*1,
            size=reps,
        )

        B = rng.poisson(
            lam*2,
            size=reps,
        )

        N1 = A
        N3 = A+B

        nested_cov = np.cov(
            N1,
            N3,
            ddof=0,
        )[0,1]

        # Superposition.
        stream1 = rng.poisson(
            3*0.5,
            size=reps,
        )

        stream2 = rng.poisson(
            7*0.5,
            size=reps,
        )

        combined = stream1+stream2

        rows = [
            "| diagnostic | empirical | theory |",
            "|---|---:|---:|",
            f"| mean N(2), lambda=2 | {empirical_mean:.4f} | 4.0000 |",
            f"| variance N(2) | {empirical_var:.4f} | 4.0000 |",
            f"| corr disjoint increments | {inc_corr:.4f} | 0 |",
            f"| mean thinned count | {kept.mean():.4f} | {p*lam*t0:.4f} |",
            f"| corr kept/removed | {thin_corr:.4f} | 0 |",
            f"| Cov(N(1),N(3)) | {nested_cov:.4f} | {lam:.4f} |",
            f"| mean superposed half-hour count | {combined.mean():.4f} | 5.0000 |",
        ]

        display(Markdown("\n".join(rows)))


lab_N.observe(update_full_poisson_lab, names="value")
display(widgets.VBox([lab_N,lab_output]))
update_full_poisson_lab()


### Conditional simulation without rejection

To simulate the arrival locations **given**

$$
N(t)=n,
$$

we need not simulate exponential waiting times and reject paths until the total count happens to equal $n$.

The theorem gives a direct exact method:

1. draw $n$ independent $U(0,t)$ variables;
2. sort them.

This is an example where a structural theorem produces a much more efficient conditional simulation algorithm.


In [ ]:
cond_reps = widgets.IntSlider(
    value=20000,
    min=2000,
    max=100000,
    step=2000,
    description="reps",
)
cond_output = widgets.Output()


def update_conditional_sim(*_):
    with cond_output:
        clear_output(wait=True)

        reps = cond_reps.value
        n = 5
        t = 10
        k = 2

        rng = np.random.default_rng(2026)

        U = np.sort(
            rng.uniform(
                0,
                t,
                size=(reps,n),
            ),
            axis=1,
        )

        sk = U[:,k-1]

        display(Math(
            r"\widehat{\mathbb E}[S_2\mid N(10)=5]="
            + f"{sk.mean():.6f}"
        ))
        display(Math(
            r"\text{theory}="
            + f"{k*t/(n+1):.6f}"
        ))


cond_reps.observe(update_conditional_sim, names="value")
display(widgets.VBox([cond_reps,cond_output]))
update_conditional_sim()


## 26. Formula summary

| Concept | Main statement |
|---|---|
| interarrivals | $X_n\stackrel{\mathrm{i.i.d.}}\sim\operatorname{Exp}(\lambda)$ |
| arrival time | $S_n\sim\operatorname{Gamma}(n,\lambda)$ |
| count law | $N(t)\sim\operatorname{Poisson}(\lambda t)$ |
| increment | $N(t)-N(s)\sim\operatorname{Poisson}(\lambda(t-s))$ |
| disjoint increments | independent |
| transition kernel | $p_{ij}(s)=e^{-\lambda s}(\lambda s)^{j-i}/(j-i)!$ for $j\ge i$ |
| conditional mean | $\mathbb E[N(t)\mid\mathcal F_s]=N(s)+\lambda(t-s)$ |
| compensated martingale | $N(t)-\lambda t$ |
| residual wait | Exp$(\lambda)$ and independent of $\mathcal F_t$ |
| covariance | $\lambda\min(s,t)$ |
| correlation | $\sqrt{\min(s,t)/\max(s,t)}$ |
| generator | $(Af)(i)=\lambda[f(i+1)-f(i)]$ |
| small time | $P(\Delta_hN=1)=\lambda h+o(h)$ |
| superposition | independent rates add |
| competing clocks | minimum rate is sum of rates |
| thinning | rates $p\lambda$ and $(1-p)\lambda$ |
| conditional arrivals | ordered $U(0,t)$ variables |
| normalized $S_k$ | Beta$(k,n+1-k)$ |
| normalized gaps | Dirichlet$(1,\ldots,1)$ |
| long-run rate | $N(t)/t\to\lambda$ a.s. |
| NHPP | $N(t)-N(s)\sim\operatorname{Poisson}(\Lambda(t)-\Lambda(s))$ |
| time change | $N(t)=\widetilde N(\Lambda(t))$ |


## 27. Guided exercise generator


In [ ]:
exercise_rng = random.Random(20260815)

exercise_kind = widgets.Dropdown(
    options=[
        ("Random","random"),
        ("Count law","count"),
        ("Arrival time","arrival"),
        ("Increment","increment"),
        ("Markov transition","markov"),
        ("Martingale","martingale"),
        ("Covariance","cov"),
        ("Generator","generator"),
        ("Thinning","thin"),
        ("Conditional arrival","conditional"),
        ("NHPP","nhpp"),
    ],
    value="random",
    description="Type",
)

new_button = widgets.Button(description="New exercise")
hint_button = widgets.Button(description="Hint")
reveal_button = widgets.Button(description="Reveal")
check_button = widgets.Button(description="Check")
answer_box = widgets.Text(description="Answer")
prompt_output = widgets.Output()
feedback_output = widgets.Output()
state = {}


def make_exercise(_=None):
    kind = exercise_kind.value

    if kind == "random":
        kind = exercise_rng.choice([
            "count",
            "arrival",
            "increment",
            "markov",
            "martingale",
            "cov",
            "generator",
            "thin",
            "conditional",
            "nhpp",
        ])

    if kind == "count":
        target = "6"
        prompt = "A rate-3 Poisson process is observed for 2 time units. What is E[N(2)]?"
        hint = "Use lambda t."
        solution = r"\mathbb E[N(2)]=6."

    elif kind == "arrival":
        target = "2"
        prompt = "For a rate-2 process, what is E[S_4]?"
        hint = "S_4 is Gamma(4,2)."
        solution = r"\mathbb E[S_4]=2."

    elif kind == "increment":
        target = "3"
        prompt = "A rate-2 process is observed on an interval of length 1.5. What is the Poisson parameter of the increment?"
        hint = "Use lambda times interval length."
        solution = r"\lambda\Delta t=3."

    elif kind == "markov":
        target = str(2*math.exp(-2))
        prompt = "Rate 2, N(3)=5. Find P(N(4)=7|N(3)=5) as a decimal."
        hint = "The future increment is Poisson(2), and two extra events are needed."
        solution = r"2e^{-2}."

    elif kind == "martingale":
        target = "yes"
        prompt = "Is N(t)-lambda t a martingale with respect to the natural filtration? yes/no"
        hint = "Subtract the deterministic mean drift."
        solution = r"\text{Yes.}"

    elif kind == "cov":
        target = "3"
        prompt = "Rate 3, s=1, t=4. Find Cov(N(1),N(4))."
        hint = "Use lambda min(s,t)."
        solution = r"\operatorname{Cov}=3."

    elif kind == "generator":
        target = "2"
        prompt = "Rate 2 and f(i)=i. What is (Af)(i)?"
        hint = "The count has instantaneous drift lambda."
        solution = r"(Af)(i)=2."

    elif kind == "thin":
        target = "3"
        prompt = "A rate-10 stream is retained independently with probability 0.3. What is the retained rate?"
        hint = "Multiply the original rate by p."
        solution = r"3."

    elif kind == "conditional":
        target = "5"
        prompt = "Given N(10)=3, what is E[S_2|N(10)=3]?"
        hint = "Use k t/(n+1)."
        solution = r"5."

    else:
        target = "8"
        prompt = "For lambda(t)=2+t, find the Poisson parameter of N(3)-N(1)."
        hint = "Integrate 2+t from 1 to 3."
        solution = r"8."

    state.clear()
    state.update(
        target=target,
        hint=hint,
        solution=solution,
    )

    answer_box.value = ""

    with prompt_output:
        clear_output(wait=True)
        display(Markdown("### Exercise\n" + prompt))

    with feedback_output:
        clear_output(wait=True)


def show_hint(_):
    with feedback_output:
        clear_output(wait=True)
        display(Markdown("**Hint:** " + state["hint"]))


def reveal(_):
    with feedback_output:
        clear_output(wait=True)
        display(Math(state["solution"]))


def check(_):
    with feedback_output:
        clear_output(wait=True)

        guess = answer_box.value.strip().lower().replace(" ","")
        target = state["target"].strip().lower().replace(" ","")

        correct = guess == target

        if not correct:
            try:
                correct = abs(float(guess)-float(target)) < 5e-4
            except Exception:
                pass

        display(Markdown(
            "**Correct.**"
            if correct
            else "**Not yet. Identify whether the question concerns counts, increments, arrival times or a conditional law.**"
        ))


new_button.on_click(make_exercise)
hint_button.on_click(show_hint)
reveal_button.on_click(reveal)
check_button.on_click(check)

display(widgets.VBox([
    widgets.HBox([exercise_kind,new_button]),
    prompt_output,
    widgets.HBox([answer_box,check_button]),
    widgets.HBox([hint_button,reveal_button]),
    feedback_output,
]))

make_exercise()


## 28. AI Audit: Poisson-process claims

Use this checklist on any AI-generated solution.

1. Is a counting process non-decreasing and right-continuous?
2. Are interarrival times distinguished from arrival times?
3. Is the rate parameter interpreted as events per unit time?
4. Is the rate-zero convention handled separately from exponential interarrivals?
5. Is non-explosion recognized as a pathwise requirement?
6. Is $\{N(t)\ge n\}=\{S_n\le t\}$ used correctly?
7. Is $S_n$ Gamma$(n,\lambda)$ in shape--rate notation?
8. Is $N(t)$ Poisson with parameter $\lambda t$?
9. Are both mean and variance of $N(t)$ equal to $\lambda t$?
10. Is the rare-event Poisson limit distinguished from finite equality?
11. Is stationarity of increments being confused with stationarity of $N(t)$?
12. Are counts on disjoint intervals distinguished from nested counts at two times?
13. Is the natural filtration understood as the complete observed past?
14. Is the future increment recognized as independent of $\mathcal F_t$?
15. Is the continuous-time Markov kernel allowed only to move upward?
16. Is the semigroup identity used in the correct time order?
17. Is $\mathbb E[N(t)\mid\mathcal F_s]=N(s)+\lambda(t-s)$?
18. Is $N(t)-\lambda t$ recognized as a martingale?
19. Is the residual waiting time fresh Exp$(\lambda)$ at a fixed observation time?
20. Is $\operatorname{Cov}(N(s),N(t))=\lambda\min(s,t)$?
21. Is $\operatorname{Corr}(N(s),N(t))$ independent of $\lambda$?
22. Is the generator $\lambda[f(i+1)-f(i)]$?
23. Are the forward equations using inflow from $n-1$ and outflow from $n$?
24. Is $P(N(h)\ge2)=o(h)$?
25. Is characteristic-function uniqueness explicitly used in the small-time characterization proof?
26. Are superposed Poisson processes assumed independent before adding rates?
27. Are competing exponential probabilities proportional to rates?
28. Is thinning performed independently event by event?
29. Are thinned subprocesses recognized as independent Poisson processes?
30. Given $N(t)=n$, are the **ordered** arrival times order statistics rather than independent uniforms?
31. Is $S_k/t$ Beta$(k,n+1-k)$ conditionally?
32. Are the conditional gaps recognized as dependent because their sum is fixed?
33. Is $N(t)/t\to\lambda$ interpreted as an almost-sure long-run frequency statement?
34. For an NHPP, is $\Lambda(t)=\int_0^t\lambda(u)\,du$ used?
35. Are NHPP increments independent but generally not stationary?
36. Is simulation being used as illustration rather than as proof of the process theorems?

### Claims to audit

- “Poisson counts at different times are independent.”
- “Stationary increments mean that $N(t)$ has the same distribution for every $t$.”
- “If $N(t)$ is Poisson$(\lambda t)$ for every $t$, then the process is automatically a Poisson process.”
- “The sum of two dependent Poisson processes is always Poisson.”
- “Given $N(t)=n$, the arrival times $S_1,\ldots,S_n$ are independent uniforms.”
- “A nonhomogeneous Poisson process has stationary increments.”

All six claims are false as written.


### Repairs

Nested counts share the arrivals that happened before the earlier time, so they are dependent.

Stationary increments concern only differences over intervals of equal length.

Correct one-dimensional Poisson marginals alone do not imply the required joint increment structure.

Superposition needs independence of the original Poisson streams.

Given the total count, the ordered arrival vector has the distribution of **order statistics** of independent uniforms. The ordered coordinates are not independent.

A nonhomogeneous process keeps independent increments but generally loses stationarity of increments.


### Suggested AI-guided activities

- “Start from independent exponential waiting times and make me derive the Gamma law of $S_n$, the dual count identity, and then the Poisson law of $N(t)$.”
- “Give me a statement about the natural filtration and make me identify exactly which future increment is independent of it.”
- “Guide me through the small-time characterization and force me to explain where characteristic-function uniqueness is used.”
- “Build one example that uses superposition, competing exponentials, thinning and conditional arrival times.”
- “Give me a nonhomogeneous intensity function and make me compute $\Lambda$, increment laws and a time-change simulation.”


## 29. Self-check quiz


In [ ]:
quiz_data = [
    (
        "1. The nth arrival time has distribution:",
        ["Choose...","Gamma(n,lambda)","Poisson(lambda n)","Normal(n,lambda)"],
        "Gamma(n,lambda)",
        r"S_n\sim\operatorname{Gamma}(n,\lambda).",
    ),
    (
        "2. N(t) has distribution:",
        ["Choose...","Poisson(lambda t)","Exp(lambda t)","Gamma(t,lambda)"],
        "Poisson(lambda t)",
        r"N(t)\sim\operatorname{Poisson}(\lambda t).",
    ),
    (
        "3. Counts N(s) and N(t), s<t, are generally independent:",
        ["Choose...","true","false"],
        "false",
        r"\operatorname{Cov}(N(s),N(t))=\lambda s.",
    ),
    (
        "4. Disjoint increments are independent:",
        ["Choose...","true","false"],
        "true",
        r"\text{Independent increments are a defining structural property.}",
    ),
    (
        "5. The compensated Poisson process is:",
        ["Choose...","N(t)-lambda t","N(t)+lambda t","N(t)/t"],
        "N(t)-lambda t",
        r"M(t)=N(t)-\lambda t.",
    ),
    (
        "6. The residual waiting time from a fixed observation time is:",
        ["Choose...","Exp(lambda)","Uniform","Gamma(2,lambda)"],
        "Exp(lambda)",
        r"R_t\sim\operatorname{Exp}(\lambda).",
    ),
    (
        "7. The Poisson generator is:",
        ["Choose...","lambda[f(i+1)-f(i)]","lambda f(i+1)","f(i)-f(i+1)"],
        "lambda[f(i+1)-f(i)]",
        r"(Af)(i)=\lambda[f(i+1)-f(i)].",
    ),
    (
        "8. Independent Poisson rates under superposition:",
        ["Choose...","add","multiply","average"],
        "add",
        r"\lambda_{\mathrm{sum}}=\lambda_1+\lambda_2.",
    ),
    (
        "9. Independent thinning with retention probability p produces retained rate:",
        ["Choose...","p lambda","lambda/p","(1-p)lambda"],
        "p lambda",
        r"\lambda_{\mathrm{keep}}=p\lambda.",
    ),
    (
        "10. Given N(t)=n, arrival times are:",
        ["Choose...","uniform order statistics","independent exponentials","independent Poisson variables"],
        "uniform order statistics",
        r"(S_1,\ldots,S_n)\stackrel d=(U_{(1)},\ldots,U_{(n)}).",
    ),
    (
        "11. N(t)/t converges almost surely to:",
        ["Choose...","lambda","1/lambda","0"],
        "lambda",
        r"N(t)/t\to\lambda.",
    ),
    (
        "12. NHPP increments are generally stationary:",
        ["Choose...","true","false"],
        "false",
        r"\text{Only the homogeneous process has stationary increments in general.}",
    ),
]

quiz_widgets = []
quiz_rows = []

for prompt, options, _, _ in quiz_data:
    dropdown = widgets.Dropdown(
        options=options,
        value="Choose...",
        layout=widgets.Layout(width="500px"),
    )

    quiz_widgets.append(dropdown)

    quiz_rows.append(widgets.HBox([
        widgets.HTML(
            f"<div style='width:720px'>{prompt}</div>"
        ),
        dropdown,
    ]))

grade_button = widgets.Button(description="Grade quiz")
quiz_output = widgets.Output()


def grade_quiz(_):
    with quiz_output:
        clear_output(wait=True)

        score = sum(
            widget.value == correct
            for widget, (_,_,correct,_) in zip(
                quiz_widgets,
                quiz_data,
            )
        )

        display(Markdown(
            f"### Score: {score}/{len(quiz_data)}"
        ))

        for i, (
            widget,
            (_,_,correct,explanation),
        ) in enumerate(
            zip(
                quiz_widgets,
                quiz_data,
            ),
            1,
        ):
            mark = "✓" if widget.value == correct else "✗"

            display(Markdown(
                f"**{mark} Question {i}:** correct answer = `{correct}`"
            ))

            display(Math(explanation))


grade_button.on_click(grade_quiz)

display(widgets.VBox(
    quiz_rows
    +
    [grade_button,quiz_output]
))


## 30. Automatic mathematical verification

The final code cell checks representative identities from all major sections.


In [ ]:
# Count law and moments.
lam = 2.0
t = 3.0
mean = lam*t

pmf_sum = sum(
    poisson_pmf(k,mean)
    for k in range(60)
)

assert abs(pmf_sum-1) < 1e-12
assert mean == 6

# Fourth arrival moments.
assert abs(4/2-2) < 1e-12
assert abs(4/2**2-1) < 1e-12

# Transition example.
assert abs(
    poisson_transition(5,7,1,2)
    -
    2*math.exp(-2)
) < 1e-12

# Semigroup example.
i,j = 2,7
s,t = 0.7,1.3
lam = 2.5

lhs = poisson_transition(i,j,s+t,lam)
rhs = sum(
    poisson_transition(i,k,s,lam)
    *
    poisson_transition(k,j,t,lam)
    for k in range(i,j+1)
)

assert abs(lhs-rhs) < 1e-12

# Conditional mean.
assert abs(
    7+5*(3.5-2)-14.5
) < 1e-12

# Covariance and correlation.
assert abs(3*min(1,4)-3) < 1e-12
assert abs(
    math.sqrt(1/4)-0.5
) < 1e-12

# Generator.
assert abs(
    poisson_generator_f(
        lambda k:k,
        4,
        2,
    )
    -
    2
) < 1e-12

assert abs(
    poisson_generator_f(
        lambda k:k*k,
        4,
        2,
    )
    -
    2*(2*4+1)
) < 1e-12

# Forward equation.
n = 3
t = 1.2
lam = 2.0
h = 1e-6

numerical = (
    poisson_pmf(n,lam*(t+h))
    -
    poisson_pmf(n,lam*(t-h))
)/(2*h)

rhs = (
    lam*poisson_pmf(n-1,lam*t)
    -
    lam*poisson_pmf(n,lam*t)
)

assert abs(numerical-rhs) < 1e-7

# Small-time two-or-more probability is quadratic.
h = 1e-5
pge2 = (
    1
    -
    poisson_pmf(0,lam*h)
    -
    poisson_pmf(1,lam*h)
)

assert pge2/h < 1e-3

# Superposition.
assert 2+5 == 7

# Competing clocks.
assert abs(3/(3+7)-0.3) < 1e-12
assert abs(1/(3+7)-0.1) < 1e-12

# Thinning.
assert abs(0.3*10-3) < 1e-12
assert abs(0.7*10-7) < 1e-12

# Conditional arrival mean.
assert abs(
    2/(5+1)*10
    -
    10/3
) < 1e-12

# Gaps.
assert abs(
    8/(3+1)-2
) < 1e-12

# NHPP with lambda(t)=2+t.
Lambda3 = 2*3+3**2/2
Lambda1 = 2*1+1**2/2

assert abs(Lambda3-10.5) < 1e-12
assert abs((Lambda3-Lambda1)-8) < 1e-12

# Rate-zero convention.
assert poisson_pmf(0,0) == 1
assert poisson_pmf(1,0) == 0

show_result(
    "All Chapter 17 automatic checks passed",
    r"N(t)\sim\operatorname{Poisson}(\lambda t)",
    r"S_n\sim\operatorname{Gamma}(n,\lambda)",
    r"\mathbb E[N(t)\mid\mathcal F_s]=N(s)+\lambda(t-s)",
    r"(Af)(i)=\lambda[f(i+1)-f(i)]",
    r"\operatorname{Cov}(N(s),N(t))=\lambda\min(s,t)",
    r"\frac{N(t)}t\to\lambda\quad\text{a.s.}",
    note=(
        "Count, arrival-time, semigroup, martingale prediction, generator, "
        "small-time, splitting and nonhomogeneous-process checks all passed."
    ),
)


## 31. Chapter map

| Chapter concept | Computational representation |
|---|---|
| counting process | step-function sample path |
| exponential interarrivals | accumulated exponential waits |
| non-explosion | finite count on every finite horizon |
| arrival/count duality | $N(t)$ versus $S_n$ |
| joint arrival density | ordered simplex |
| Gamma arrival law | density and moments |
| Poisson count law | pmf and histogram |
| rare-event limit | Binomial-to-Poisson comparison |
| Erlang application | call arrivals and fifth waiting time |
| stationary increments | interval-length dependence |
| independent increments | near-zero empirical correlation |
| natural filtration | complete observed past |
| future/past independence | future increment versus $\mathcal F_t$ |
| Markov transition kernel | conditional future count |
| semigroup | transition convolution |
| conditional mean | observed count plus future mean |
| compensated martingale | $N(t)-\lambda t$ |
| residual wait | fresh exponential |
| covariance | nested count decomposition |
| generator | one-jump infinitesimal dynamics |
| forward equations | inflow/outflow ODE |
| small-time rule | zero/one/multiple event scaling |
| equivalent characterizations | CF differential equation |
| superposition | rates add |
| competing exponentials | summed clock rate |
| thinning | independent retained/discarded streams |
| multitype splitting | independent type rates |
| conditional arrival times | sorted uniforms |
| conditional Beta law | $S_k/t$ |
| conditional gaps | Dirichlet simplex |
| strong long-run rate | $N(t)/t$ |
| nonhomogeneous process | cumulative intensity |
| time change | unit-rate process at clock $\Lambda(t)$ |
| AI Audit | structural-process checks |

The chapter's main architecture is:

$$
\boxed{
\text{waiting times}
\longleftrightarrow
\text{counts}
\longleftrightarrow
\text{increments}
\longleftrightarrow
\text{small-time rules}.
}
$$

The same process can be analyzed through whichever representation is most useful for the question at hand.
